Cellule 1 : Connexion à DuckDB et chargement des données

# SolarOps Maroc : Modélisation et MLOps

**Objectif :** Entraîner un modèle de prévision de puissance solaire (Random Forest) en corrigeant les limites identifiées au Jour 3.
- Utilisation de `pvlib` pour calculer la position réelle du soleil (Time Center).
- Découpage temporel strict (Train 2015-2023 / Test 2024).
- Tracking des expériences avec **MLflow**.
- Évaluation avec le **Skill Score** face à la baseline de persistance.

In [3]:
!pip install --upgrade duckdb


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import pvlib
from pvlib.location import Location
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Lecture directe du fichier Parquet (beaucoup plus rapide et robuste !)
df = pd.read_parquet('/app/data/fct_solar_features.parquet')

# 2. Formatage des dates
df['time_utc'] = pd.to_datetime(df['time_utc'])
df = df.sort_values('time_utc').reset_index(drop=True)

print(f"✅ Lignes chargées depuis le fichier Parquet : {len(df)}")
print(df[['city', 'time_utc', 'shortwave_radiation']].head(3))

✅ Lignes chargées depuis le fichier Parquet : 438360
         city   time_utc  shortwave_radiation
0  Al Hoceima 2015-01-01                  0.0
1  Ouarzazate 2015-01-01                  0.0
2    Laayoune 2015-01-01                  0.0


Feature Engineering temporel (Le correctif)

In [5]:
# 1. Création du "Time Center" (on recule de 30 min : 13h00 -> 12h30)
df['time_center'] = df['time_utc'] - pd.Timedelta(minutes=30)
df['time_center'] = df['time_center'].dt.tz_localize('UTC')

# 2. Coordonnées GPS des villes
coords = {
    'Ouarzazate': (30.9335, -6.9370),
    'Laayoune': (27.1253, -13.1625),
    'Dakhla': (23.6848, -15.9580),
    'Tanger': (35.7595, -5.8340),
    'Al Hoceima': (35.2517, -3.9372)
}

# 3. Calcul de l'angle zénithal du soleil avec pvlib
df['solar_zenith'] = 0.0

for city, (lat, lon) in coords.items():
    mask = df['city'] == city
    loc = Location(lat, lon, tz='UTC')
    # pvlib calcule la position exacte du soleil pour chaque heure
    solar_pos = loc.get_solarposition(df.loc[mask, 'time_center'])
    df.loc[mask, 'solar_zenith'] = solar_pos['zenith'].values

# 4. Création d'une feature "proxy ciel clair" basée sur la physique (et non sur l'observé)
# Plus le soleil est haut (zénith proche de 0), plus cos(zénith) est proche de 1.
df['clear_sky_proxy'] = np.cos(np.radians(df['solar_zenith']))
df['clear_sky_proxy'] = df['clear_sky_proxy'].clip(lower=0) # Nul la nuit

print("✅ Features physiques 'solar_zenith' et 'clear_sky_proxy' ajoutées.")

✅ Features physiques 'solar_zenith' et 'clear_sky_proxy' ajoutées.


Découpage Train / Test & Baseline

In [6]:
# Découpage Temporel Strict : Entraînement (2015-2023) et Test (2024)
train = df[df['time_utc'] < '2024-01-01'].copy()
test = df[df['time_utc'] >= '2024-01-01'].copy()

# Liste des features (on ajoute nos variables physiques)
features = [
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'radiation_lag_24h',
    'solar_zenith', 'clear_sky_proxy', # <-- Les variables pvlib
    'temperature_2m', 'relative_humidity_2m', 'surface_pressure',
    'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high',
    'wind_speed_10m', 'wind_direction_10m'
]
target = 'target_power_kw'

# On supprime les NaN (les premières 24h de 2015 n'ont pas de lag J-1)
train = train.dropna(subset=features)
test = test.dropna(subset=features)

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

# Calcul de la Baseline (Persistance J-1)
y_baseline = test['radiation_lag_24h'] * 0.85
mae_baseline = mean_absolute_error(y_test, y_baseline)

print(f"✅ Train : {len(X_train)} lignes (2015-2023)")
print(f"✅ Test  : {len(X_test)} lignes (2024)")
print(f"🎯 MAE Baseline (Persistance J-1) : {mae_baseline:.2f} kW")

✅ Train : 394320 lignes (2015-2023)
✅ Test  : 43920 lignes (2024)
🎯 MAE Baseline (Persistance J-1) : 20.35 kW


Entraînement et MLflow

In [8]:
# Configuration de MLflow
# On force MLflow à utiliser le dossier /app/mlruns
mlflow.set_tracking_uri("file:///app/mlruns")
mlflow.set_experiment("SolarOps_Prevision")

with mlflow.start_run(run_name="RandomForest_pvlib_v1"):
    print("⏳ Entraînement du modèle en cours...")
    
    # 1. Initialisation et entraînement
    model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    
    # 2. Prédictions
    y_pred = model.predict(X_test)
    
    # 3. Calcul des métriques
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    skill_score = 1 - (mae / mae_baseline)
    
    print(f"✅ MAE Modèle : {mae:.2f} kW")
    print(f"✅ RMSE Modèle : {rmse:.2f} kW")
    print(f"🏆 Skill Score (vs Baseline) : {skill_score:.2%}")
    
    # 4. Enregistrement dans MLflow
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("features", len(features))
    
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("Skill_Score", skill_score)
    
    # Sauvegarde du modèle dans MLflow
    mlflow.sklearn.log_model(model, "random_forest_model")
    
    print("💾 Modèle et métriques sauvegardés dans MLflow !")

2026/11/23 20:39:41 INFO mlflow.tracking.fluent: Experiment with name 'SolarOps_Prevision' does not exist. Creating a new experiment.


⏳ Entraînement du modèle en cours...
✅ MAE Modèle : 11.57 kW
✅ RMSE Modèle : 28.41 kW
🏆 Skill Score (vs Baseline) : 43.13%


/usr/local/lib/python3.11/site-packages/_distutils_hack/__init__.py:15: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
/usr/local/lib/python3.11/site-packages/_distutils_hack/__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


💾 Modèle et métriques sauvegardés dans MLflow !
